# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [23]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('./netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [24]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [25]:
# Task 1

# ── Data prep ─────────────────────────────────────────────────────────────────
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

ratings_keep = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df.loc[df['rating'].isin(ratings_keep)]

rating_decade = df_filtered.groupby(['rating', 'decade']).size().reset_index(name='count')
pivot = rating_decade.pivot(index='rating', columns='decade', values='count').fillna(0)

# ── Chart ─────────────────────────────────────────────────────────────────────
fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',       # sequential: low count → high count
    text_auto=True,                        # show count value inside each cell
    aspect='auto',
    width=1000, height=700
)

# ── Customisation ─────────────────────────────────────────────────────────────
fig.update_traces(
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

fig.update_layout(
    title='TV-MA and TV-14 dominate the 2010s — mature content surged in the streaming era',
    font=dict(family='Arial', size=11),
    coloraxis_colorbar=dict(title='Titles'),
    margin=dict(l=80, r=40, t=55, b=60),
    xaxis=dict(title='Release Decade'),
    yaxis=dict(title='Rating')
)

fig.update_xaxes(title='Decade')
fig.update_yaxes(title='')

fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [26]:
# Task 2

# ── Data prep ─────────────────────────────────────────────────────────────────
movies = df.loc[df['type'] == 'Movie']
adds = movies.groupby('added_year').size().reset_index(name='new_titles')
adds = adds.loc[adds['added_year'].between(2015, 2022)].copy()

cumulative = adds['new_titles'].sum()

# Find the year with the largest single addition for annotation
peak_idx = adds['new_titles'].idxmax()
peak_year = str(adds.loc[peak_idx, 'added_year'])
peak_val  = adds.loc[peak_idx, 'new_titles']

# ── Chart ─────────────────────────────────────────────────────────────────────
trace = go.Waterfall(
    x=adds['added_year'].astype(str).tolist() + ['Total 2015–2022'],
    y=adds['new_titles'].tolist() + [cumulative],
    measure=['relative'] * len(adds) + ['total'],
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),    # green for yearly additions
    totals=dict(marker_color='#2E75B6'),         # blue for cumulative total
    texttemplate='%{y:,}',
    textposition='outside',
)

fig = go.Figure(data=[trace])

# ── Annotation on the peak year ───────────────────────────────────────────────
fig.add_annotation(
    x=peak_year,
    y=peak_val + 15,
    text=f'Peak year: {peak_val} titles',
    showarrow=True,
    arrowhead=2,
    arrowcolor='#E15759',
    font=dict(color='#E15759', size=11),
    ax=0, ay=-40
)

# ── Customisation ─────────────────────────────────────────────────────────────
fig.update_layout(
    title='Netflix steadily grew its Movie library — 659 films added between 2015 and 2022',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=60, r=40, t=55, b=40),
    height=600,
)
fig.update_xaxes(title='Year', showgrid=False)
fig.update_yaxes(title='Movies Added', gridcolor='#EEEEEE')

fig.show()
